# Whitebox Demo

This notebook demonstrates all major features of Whitebox - a model-agnostic visualization library for interpreting ML models by comparing GLM and GBM.

**Contents:**
1. Setup & Data Generation
2. Initialize Whitebox
3. SHAP Value Computation
4. Univariate Plots
5. Bivariate Plots
6. Model Comparison
7. Configuration Customization
8. Plot Engines (Bokeh vs Matplotlib)

## 1. Setup & Data Generation

First, let's import the required packages and generate synthetic insurance data.

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Bokeh setup for notebook display
from bokeh.io import output_notebook
output_notebook()

# Import demo modules
from synthetic_data import generate_synthetic_data, train_xgboost_model, prepare_data_for_whitebox
from glm_helpers import get_glm_data_for_whitebox

# Import Whitebox
from whitebox import Whitebox

print("All imports successful!")

In [2]:
# Generate synthetic insurance data
raw_data = generate_synthetic_data(n_samples=10000, random_state=42)

print(f"Generated {len(raw_data):,} samples")
print(f"\nColumns: {list(raw_data.columns)}")
raw_data.head()

Generated 10,000 samples

Columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type', 'exposure', 'claim_count']


,age,vehicle_value,years_licensed,region,vehicle_type,exposure,claim_count
0,52.450712,15689.588755,11.677637,North,SUV,0.639255,0
1,42.926035,18906.296023,24.677705,West,Sedan,0.946568,0
2,54.715328,16338.988603,31.575657,South,SUV,0.989004,0
3,67.845448,23276.720582,45.267958,South,Truck,0.839560,0
4,41.487699,40078.257759,18.461473,West,SUV,0.985150,0


In [3]:
raw_data.dtypes


age               float64
vehicle_value     float64
years_licensed    float64
region                str
vehicle_type          str
exposure          float64
claim_count         int64
dtype: object

## Categorical Variable Encoding for Whitebox

### Why Encoding is Required

GBM models (XGBoost, LightGBM) require **numeric input**, so categorical variables must be encoded as integers. However, for interpretable plots, we need to display **human-readable labels**. Whitebox bridges this gap using `category_mappings`.

### ⚠️ CRITICAL: The `_encoded` Suffix Convention

Whitebox expects encoded columns to follow a **strict naming convention**:

```
{original_column_name}_encoded
```

**Examples:**
- `region` → `region_encoded`
- `vehicle_type` → `vehicle_type_encoded`

**Why this matters:**
- Whitebox automatically detects `_encoded` columns for SHAP calculations
- The GBM model uses `_encoded` columns internally
- Plot labels are generated from the original column + `category_mappings`

### Required Data Structure

For each categorical variable, your DataFrame must have:

| Column | Type | Purpose | Example Values |
|--------|------|---------|----------------|
| `region` | string | Original labels for display | `"North"`, `"South"`, `"East"`, `"West"` |
| `region_encoded` | int | Numeric codes for GBM (must use `_encoded` suffix!) | `0`, `1`, `2`, `3` |

### The `category_mappings` Dictionary

This dictionary maps numeric codes back to labels:

```python
category_mappings = {
    'region': {0: 'East', 1: 'North', 2: 'South', 3: 'West'},
    'vehicle_type': {0: 'SUV', 1: 'Sedan', 2: 'Sports', 3: 'Truck'}
}
```

**Key format:** `{column_name: {code: label, ...}}`

**Important:** Keys in `category_mappings` use the **original column name** (e.g., `'region'`), not the encoded name.

### Option 1: Use `prepare_data_for_whitebox()` (Recommended)

```python
from synthetic_data import prepare_data_for_whitebox

data, category_mappings = prepare_data_for_whitebox(
    raw_data, 
    feature_names,
    optimize=True  # Optional: reduce memory usage
)
```

This automatically:
- Detects string/object columns in `feature_names`
- Creates `{col}_encoded` columns with the correct suffix
- Generates the `category_mappings` dictionary

### Option 2: Manual Encoding

If you have custom encoding requirements, **ensure you use the `_encoded` suffix**:

```python
# 1. Create encoded column WITH THE _encoded SUFFIX
data['region_encoded'] = data['region'].astype('category').cat.codes

# 2. Build category_mappings manually
category_mappings = {
    'region': dict(enumerate(data['region'].astype('category').cat.categories))
}
# Result: {0: 'East', 1: 'North', 2: 'South', 3: 'West'}
```

### Plot Labels: "Label(code)" Format

With `category_mappings`, plots display labels in `"Label(code)"` format:
- `"North(1)"` instead of just `1` or `"North"`
- Makes it easy to trace values back to both the model input and business meaning

In [ ]:
# Define feature names (these are the ORIGINAL column names, used for plotting)
feature_names = ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type']

# Prepare data for modeling:
# - Original columns keep human-readable values (for GLM and plot labels)
# - New {col}_encoded columns contain numeric codes (for GBM/XGBoost)
data, category_mappings = prepare_data_for_whitebox(raw_data, feature_names, optimize=True)

print("=" * 70)
print("KEY CONCEPT: GBM vs GLM use different column formats")
print("=" * 70)

print("\n📊 Category mappings (code -> label):")
for col, mapping in category_mappings.items():
    print(f"  {col}: {mapping}")

print("\n" + "-" * 70)
print("GBM (XGBoost) uses ENCODED columns with numeric codes:")
print("-" * 70)
print(data[['region_encoded', 'vehicle_type_encoded']].head())

print("\n" + "-" * 70)
print("GLM uses ORIGINAL columns with string labels:")
print("-" * 70)
print(data[['region', 'vehicle_type']].head())

print("\n✅ Whitebox handles the mapping automatically via category_mappings")

In [5]:
# Show the complete data structure
print("Complete DataFrame structure:")
print(f"Columns: {list(data.columns)}")
print(f"\nData types:")
print(data.dtypes)
print(f"\nSample of categorical columns (original vs encoded):")
data[['region', 'region_encoded', 'vehicle_type', 'vehicle_type_encoded']].head(10)

Complete DataFrame structure:
Columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type', 'exposure', 'claim_count', 'region_encoded', 'vehicle_type_encoded']

Data types:
age                     float32
vehicle_value           float64
years_licensed          float32
region                      str
vehicle_type                str
exposure                float32
claim_count                int8
region_encoded             int8
vehicle_type_encoded       int8
dtype: object

Sample of categorical columns (original vs encoded):


,region,region_encoded,vehicle_type,vehicle_type_encoded
0,North,1,SUV,0
1,West,3,Sedan,1
2,South,2,SUV,0
3,South,2,Truck,3
4,West,3,SUV,0
5,South,2,Sedan,1
6,West,3,Sedan,1
7,West,3,Truck,3
8,North,1,Sedan,1
9,North,1,Sedan,1


In [6]:
# Train XGBoost model with Poisson objective
# NOTE: train_xgboost_model automatically uses _encoded columns for categoricals
model = train_xgboost_model(
    data,
    feature_names,  # Pass original names - function uses _encoded columns internally
    weight_col='exposure',
    target_col='claim_count',
    random_state=42,
    max_depth=4,
    eta=0.1
)

print("=" * 70)
print("XGBoost Model Training")
print("=" * 70)
print("\nModel uses ENCODED columns internally:")
print("  - region_encoded (0, 1, 2, 3) instead of ('East', 'North', 'South', 'West')")
print("  - vehicle_type_encoded (0, 1, 2, 3) instead of ('SUV', 'Sedan', 'Sports', 'Truck')")
print("\nFeature importance:")
importance = model.get_score(importance_type='gain')
for feat, score in sorted(importance.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {score:.2f}")

XGBoost Model Training

Model uses ENCODED columns internally:
  - region_encoded (0, 1, 2, 3) instead of ('East', 'North', 'South', 'West')
  - vehicle_type_encoded (0, 1, 2, 3) instead of ('SUV', 'Sedan', 'Sports', 'Truck')

Feature importance:
  age: 2.06
  years_licensed: 1.42
  vehicle_value: 1.32
  vehicle_type: 1.15
  region: 1.05


## 2. Initialize Whitebox

Create a Whitebox instance with our model and data. We'll also set up simulated GLM relativities for comparison.

In [ ]:
# Create simulated GLM relativities and predictions
# NOTE: GLM uses the ENCODED columns internally but maps back to original labels
glm_df, glm_preds = get_glm_data_for_whitebox(data, feature_names)

# Add GLM predictions to data
data['glm_predictions'] = glm_preds

print("=" * 70)
print("GLM Relativities (simulated from Emblem-style model)")
print("=" * 70)
print(f"\nGLM columns: {list(glm_df.columns)}")
print(f"GLM predictions - Mean: {glm_preds.mean():.4f}, Actual Mean: {data['claim_count'].mean():.4f}")

print("\n" + "-" * 70)
print("GLM relativities for categorical variables (using encoded values):")
print("-" * 70)
print(f"Region unique GLM values: {glm_df['region'].unique()}")
print(f"Vehicle Type unique GLM values: {glm_df['vehicle_type'].unique()}")

In [8]:
glm_df.head()

,age,vehicle_value,years_licensed,region,vehicle_type
0,-0.026164,-0.051856,-0.030197,0.00,0.10
1,0.031476,-0.036936,-0.264199,0.05,0.00
2,-0.039869,-0.048611,-0.388362,0.10,0.10
3,-0.119329,-0.020299,-0.634823,0.10,-0.05
4,0.040181,0.023171,-0.152307,0.05,0.10


In [ ]:
# Initialize Whitebox with category_mappings to bridge GBM and GLM
# 
# KEY: category_mappings tells Whitebox how to:
# 1. Use _encoded columns for GBM SHAP calculations
# 2. Map numeric codes back to labels for plotting
# 3. Match GLM relativities with GBM features

print("=" * 70)
print("Initializing Whitebox with category_mappings")
print("=" * 70)

wb = Whitebox(
    data=data,
    model=model,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,           # Original names (not _encoded)
    link_fn='poisson',
    glm_preds_col='glm_predictions',
    glm_df=glm_df,
    category_mappings=category_mappings,   # Maps codes <-> labels
    verbose=True
)

print("\n" + "-" * 70)
print("Verification:")
print("-" * 70)
print(f"category_mappings stored: {wb.category_mappings}")
print(f"fac_mapping (for plot labels): {wb.fac_mapping}")

In [10]:
data.head()

,age,vehicle_value,years_licensed,region,vehicle_type,exposure,claim_count,region_encoded,vehicle_type_encoded,glm_predictions
0,52.450714,15689.588755,11.677636,North,SUV,0.639255,0.0,1,0,0.076083
1,42.926037,18906.296023,24.677706,West,Sedan,0.946568,0.0,3,1,0.091188
2,54.715328,16338.988603,31.575657,South,SUV,0.989004,0.0,2,0,0.089980
3,67.845451,23276.720582,45.267960,South,Truck,0.839560,0.0,2,3,0.048821
4,41.487698,40078.257759,18.461473,West,SUV,0.985150,0.0,3,0,0.125659


## 3. SHAP Value Computation

Whitebox computes SHAP values lazily when needed. Let's precompute them to see the process.

In [ ]:
# Compute SHAP values (this is also done automatically when plotting)
wb.DataPrep.prep_shap_values()

print("SHAP values computed and cached.")
print(f"\nSHAP DataFrame shape: {wb.shap_df.shape}")
print(f"SHAP columns: {list(wb.shap_df.columns)}")

## 4. Univariate Plots

Univariate plots show how individual features affect model predictions.

### 4.1 Basic Univariate with SHAP

In [ ]:
# Basic univariate plot showing SHAP values for age
wb.univariate_plot(
    var_name='age',
    shap=True,
    weight=True,
    plot_name='Age - Basic SHAP Plot'
)

### 4.2 SHAP with Points and Standard Deviation

In [ ]:
# SHAP with individual points and standard deviation bands
wb.univariate_plot(
    var_name='age',
    shap=True,
    shap_points=True,
    shap_sd=True,
    n_shap_points=500,
    weight=True,
    plot_name='Age - SHAP with Points and SD'
)

### 4.3 SHAP with GLM Relativities Overlay

In [ ]:
# Compare GBM SHAP values with GLM relativities
wb.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    weight=True,
    plot_name='Age - SHAP vs GLM Comparison'
)

### 4.4 Full Plot with Actuals

In [ ]:
# Complete plot with SHAP, GLM, actuals, and weights
wb.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Age - Complete Analysis'
)

### 4.5 Custom Binning

In [ ]:
# Custom binning with start, finish, and stepsize
wb.univariate_plot(
    var_name='age',
    shap=True,
    weight=True,
    start=20,
    finish=70,
    stepsize=5,
    infinity_lower=True,
    infinity_higher=True,
    plot_name='Age - Custom Binning (20-70, step=5)'
)

In [ ]:
# Custom binning with nlevels (automatic step calculation)
wb.univariate_plot(
    var_name='vehicle_value',
    shap=True,
    glm=True,
    weight=True,
    #glmindic_cols=['vehicle_value'],
    nlevels=8,
    percentile_start=5,
    percentile_finish=95,
    plot_name='Vehicle Value - 8 Levels with Percentile Bounds'
)

### 4.6 Categorical Variables - GBM vs GLM Comparison

This is where the encoding really matters:
- **GBM SHAP**: Computed using `region_encoded` (numeric codes), then mapped back to labels
- **GLM Relativities**: Computed using `region_encoded`, displayed with string labels
- **X-axis labels**: Show human-readable values from `category_mappings`

In [ ]:
# Region - categorical variable
# X-axis shows: East, North, South, West (from category_mappings)
# GBM internally uses: 0, 1, 2, 3 (from region_encoded)
# GLM relativities are mapped to the same labels

print("Data check:")
print(f"  data['region'] dtype: {data['region'].dtype} (string labels for plotting)")
print(f"  data['region_encoded'] dtype: {data['region_encoded'].dtype} (numeric codes for GBM)")
print()

wb.univariate_plot(
    var_name='region',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Region - GBM (SHAP) vs GLM Comparison'
)

In [ ]:
# Vehicle type - categorical variable  
# Same principle: GBM uses encoded, plots show labels

print("Data check:")
print(f"  data['vehicle_type'] dtype: {data['vehicle_type'].dtype} (string labels)")
print(f"  data['vehicle_type_encoded'] dtype: {data['vehicle_type_encoded'].dtype} (numeric codes)")
print(f"  Mapping: {category_mappings['vehicle_type']}")
print()

wb.univariate_plot(
    var_name='vehicle_type',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Vehicle Type - GBM (SHAP) vs GLM Comparison'
)

### 4.7 Rebasing to a Specific Level

In [ ]:
# Set a specific base level for relativities
wb.univariate_plot(
    var_name='vehicle_type',
    shap=True,
    glm=True,
    weight=True,
    base='Sedan',  # Use Sedan (its encoded but it reuses the original labels)) as base
    rebase=True,
    plot_name='Vehicle Type - Rebased to Sedan'
)

## 5. Bivariate Plots

Bivariate plots show interactions between two features.

### 5.1 Two Continuous Variables

In [ ]:
# Age x Vehicle Value interaction
wb.bivariate_plot(
    var1='age',
    var2='vehicle_value',
    shap=True,
    nlevels_var1=6,
    nlevels_var2=4,
    plot_title='Age x Vehicle Value Interaction'
)

### 5.2 Continuous x Categorical

In [ ]:
# Age x Region interaction
wb.bivariate_plot(
    var1='age',
    var2='region',
    shap=True,
    nlevels_var1=8,
    plot_title='Age x Region Interaction'
)

In [ ]:
# Vehicle Value x Vehicle Type interaction
wb.bivariate_plot(
    var1='vehicle_value',
    var2='vehicle_type',
    shap=True,
    nlevels_var1=6,
    plot_title='Vehicle Value x Vehicle Type Interaction'
)

### 5.3 Bivariate with GLM

In [ ]:
# Age x Region with GLM comparison
wb.bivariate_plot(
    var1='age',
    var2='region',
    shap=True,
    glm=True,
    nlevels_var1=6,
    plot_title='Age x Region - SHAP vs GLM'
)

## 6. Model Comparison

Compare multiple models using the `compare()` method.

In [25]:
# Train a second model with different hyperparameters
model2 = train_xgboost_model(
    data,
    feature_names,
    weight_col='exposure',
    target_col='claim_count',
    random_state=123,
    max_depth=6,  # Deeper trees
    eta=0.05      # Lower learning rate
)

print("Second model trained!")

Second model trained!


In [ ]:
# Create second Whitebox instance with category_mappings
wb2 = Whitebox(
    data=data,
    model=model2,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,
    link_fn='poisson',
    category_mappings=category_mappings,  # Use category_mappings, not mapping_dict
    verbose=False
)

print("Second Whitebox instance created!")

In [ ]:
# Compare models on age variable
wb.compare(
    wblist=[wb2],
    model_names=['Model 1 (depth=4)', 'Model 2 (depth=6)'],
    var_name='age',
    shap=True,
    weight=True,
    plot_name='Model Comparison - Age'
)

In [ ]:
# Compare models on vehicle_type
wb.compare(
    wblist=[wb2],
    model_names=['Model 1', 'Model 2'],
    var_name='vehicle_type',
    shap=True,
    weight=True,
    plot_name='Model Comparison - Vehicle Type'
)

## 7. Configuration Customization

Customize plot appearance through the `config` dictionary.

In [ ]:
# View default configuration
print("Default configuration:")
import json
print(json.dumps(wb.config, indent=2))

In [ ]:
# Create a copy of wb with custom colors
wb_custom = Whitebox(
    data=data,
    model=model,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,
    link_fn='poisson',
    glm_preds_col='glm_predictions',
    glm_df=glm_df,
    category_mappings=category_mappings,  # Use category_mappings
    shap_df=wb.shap_df,  # Reuse computed SHAP values
    verbose=False
)

# Customize colors
wb_custom.config['colors']['shap'] = '#e41a1c'       # Red
wb_custom.config['colors']['glm'] = '#4daf4a'        # Green
wb_custom.config['colors']['weight'] = '#984ea3'     # Purple
wb_custom.config['colors']['actuals'] = '#ff7f00'    # Orange

# Customize labels
wb_custom.config['labels']['shap'] = 'GBM Effect'
wb_custom.config['labels']['glm'] = 'GLM Factor'

# Customize line widths
wb_custom.config['line_width']['shap'] = 3
wb_custom.config['line_width']['glm'] = 3

print("Custom configuration applied!")

In [ ]:
# Plot with custom styling
wb_custom.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Age - Custom Styled Plot'
)

## 8. Plot Engines

Whitebox supports two plot engines: Bokeh (interactive) and Matplotlib (static).

### 8.1 Bokeh Engine (Interactive)

In [ ]:
# Bokeh is the default engine - produces interactive HTML plots
wb.univariate_plot(
    var_name='years_licensed',
    shap=True,
    glm=True,
    weight=True,
    engine='Bokeh',
    plot_name='Years Licensed - Bokeh (Interactive)'
)

### 8.2 Matplotlib Engine (Static)

In [ ]:
# Matplotlib produces static plots - useful for reports and PDFs
wb.univariate_plot(
    var_name='years_licensed',
    shap=True,
    glm=True,
    weight=True,
    engine='Matplotlib',
    plot_name='Years Licensed - Matplotlib (Static)'
)

## Summary

This demo covered:

1. **Data Generation**: Creating synthetic insurance data with realistic feature relationships
2. **Model Training**: XGBoost with Poisson objective for claim frequency modeling
3. **Whitebox Initialization**: Setting up the library with models, data, and GLM comparisons
4. **Univariate Plots**: Various ways to visualize single-feature effects
5. **Bivariate Plots**: Visualizing feature interactions
6. **Model Comparison**: Comparing multiple GBM models side by side
7. **Customization**: Changing colors, labels, and styling
8. **Plot Engines**: Using Bokeh for interactive plots and Matplotlib for static output

For more details, see the Whitebox documentation and the CLAUDE.md file in the repository root.